# PCA features

**Iteration 5** — representation learning, added after the iteration-4 round.

Linear compression of the **internal** feature space: the top principal components plus the **reconstruction error** (how far each applicant sits from the internal-data manifold — an anomaly signal a GBM can't build itself). Unsupervised, internal only, prefixed `x_pca_`; saved to `pca.pkl`.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from src.data import load_master, save_features

X, _ = load_master(all_rows=True)
idx = X.index
drop = [c for c in X.columns if c.lower().startswith(("ext_", "x_kmeans_", "x_pca_", "x_ae_")) or "ext_source" in c.lower()]
internal = X.drop(columns=drop)
del X                                               # free the full master immediately
keep = internal.columns[(internal.std(numeric_only=True) > 0) & (internal.isna().mean() < 0.9)]
internal = internal[keep]

def prep(df):
    df = df.astype("float32")
    med = df.median().astype("float32")             # keep everything float32 - no float64 upcast
    mean = df.mean().astype("float32")
    std = df.std().astype("float32").replace(0, 1)
    df = df.fillna(med)
    return ((df - mean) / std).fillna(0).to_numpy("float32")

Z = prep(internal)
del internal                                        # free the frame; keep only the float32 array
out = pd.DataFrame(index=idx)
Z.shape

(356255, 3224)

## Components + reconstruction error

In [2]:
K = 30
n = Z.shape[0]
fit_idx = np.random.default_rng(0).choice(n, size=min(80_000, n), replace=False)
pca = PCA(n_components=K, svd_solver="randomized", random_state=0).fit(Z[fit_idx])

mean_ = pca.mean_.astype("float32")
V = pca.components_.astype("float32")                 # K x n_features
comps = np.empty((n, K), dtype="float32")
recon = np.empty(n, dtype="float32")
for s in range(0, n, 20_000):                          # stream in row batches -> flat memory
    zc = Z[s:s + 20_000] - mean_
    c = zc @ V.T
    comps[s:s + 20_000] = c
    recon[s:s + 20_000] = ((zc ** 2).sum(1) - (c ** 2).sum(1)) / Z.shape[1]

for i in range(K):
    out[f"x_pca_{i}"] = comps[:, i]
out["x_pca_recon_error"] = recon                       # residual via orthogonal projection, no dense rebuild
print(f"{K} components explain {pca.explained_variance_ratio_.sum():.1%} of variance")
out.shape

30 components explain 47.5% of variance


(356255, 31)

# Save

In [3]:
save_features(out, "pca"); out.shape

(356255, 31)